# 03 · Post-training pitch-rate scaling
Nominal pitch rate $\dot\beta = \kappa_t\,\Delta\beta / T_{s,\mathrm{test}}$ with $\Delta\beta = 0.010^\circ$ and $T_{s,\mathrm{test}} = 0.025$ s (0.4 °/s for $\kappa_t = 1$).

* Fixed-rate sweeps in the 10 m/s transition profile and in the turbulent scenarios (data of Fig. 5).
* **Adaptive rule (two levels):** 0.8 °/s when $|P - P_{\mathrm{rated}}| > 0.1$ MW, otherwise 0.3 °/s; activity of each level.
* Table 5 (`tab:best`): adaptive rule in the full-load and turbulent scenarios.
* Diagnostic test at 10 m/s (0.004 °/s with a −1° pitch offset below rated), quoted only in the discussion.

Analysis window: 200–5200 s.

In [1]:
import numpy as np
import pandas as pd
from utils import load_fast_file, window, compare_to_rosco, CHANNELS, P_RATED, FIG_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

ADAPTIVE = "0.8/0.3 °/s (adaptive)"
THRESHOLD = 100.0  # kW, power deviation that triggers the fast rate

# scenario -> ROSCO file, fixed-rate DDQN files {rate [°/s]: file}, adaptive-rule file
SWEEPS = {
    "10 m/s sawtooth": {
        "rosco": "val_saw10_rosco.out",
        "fixed": {0.004: "val_saw10_ddqn_0.004dps.out", 0.04: "val_saw10_ddqn_0.04dps.out",
                  0.2: "val_saw10_ddqn_0.2dps.out", 0.3: "val_saw10_ddqn_0.3dps.out",
                  0.4: "val_saw10_ddqn_0.4dps.out", 0.6: "val_saw10_ddqn_0.6dps.out"},
        "adaptive": None,  # not applied: no power/load trade-off in the transition profile
    },
    "15 m/s sawtooth": {
        "rosco": "val_saw15_rosco.out",
        "fixed": {},
        "adaptive": "val_saw15_ddqn_0.8-0.3dps.out",
    },
    "11 m/s TI=10%": {
        "rosco": "val_11ti10_rosco.out",
        "fixed": {0.2: "val_11ti10_ddqn_0.2dps.out", 0.3: "val_11ti10_ddqn_0.3dps.out",
                  0.4: "val_11ti10_ddqn_0.4dps.out", 0.6: "val_11ti10_ddqn_0.6dps.out",
                  0.8: "val_11ti10_ddqn_0.8dps.out"},
        "adaptive": "val_11ti10_ddqn_0.8-0.3dps.out",
    },
    "15 m/s TI=10%": {
        "rosco": "val_15ti10_rosco.out",
        "fixed": {0.2: "val_15ti10_ddqn_0.2dps.out", 0.4: "val_15ti10_ddqn_0.4dps.out",
                  0.8: "val_15ti10_ddqn_0.8dps.out"},
        "adaptive": "val_15ti10_ddqn_0.8-0.3dps.out",
    },
}

# Diagnostic test (discussion only): 0.004 °/s with a -1° pitch offset for v < v_rated
DIAGNOSTIC = "val_saw10_ddqn_0.004dps_off-1-below.out"

In [2]:
# Run all comparisons once
tables, adaptive_runs = {}, {}
for scenario, cfg in SWEEPS.items():
    rosco = load_fast_file(cfg["rosco"])[0]
    cases = {f"{rate} °/s": load_fast_file(f)[0] for rate, f in cfg["fixed"].items()}
    if cfg["adaptive"]:
        adaptive_runs[scenario] = load_fast_file(cfg["adaptive"])[0]
        cases[ADAPTIVE] = adaptive_runs[scenario]
    tables[scenario] = compare_to_rosco(rosco, cases)

## 10 m/s sawtooth (transition): fixed rates

In [3]:
tables["10 m/s sawtooth"]

,ROSCO,0.004 °/s,0.04 °/s,0.2 °/s,0.3 °/s,0.4 °/s,0.6 °/s
GenPwr mean,3229.896,3371.980,3370.216,3371.064,3371.662,3377.439,3374.485
GenPwr scaled STD,0.367,0.126,0.137,0.187,0.218,0.247,0.301
TwrBsMyt mean,51507.237,55603.937,55489.167,55300.000,55228.038,55367.676,55371.198
TwrBsMyt scaled STD,0.269,0.304,0.357,0.597,0.745,0.893,1.169
RootMyb1 mean,7492.026,8020.954,8005.425,7982.755,7974.994,7997.330,7997.771
RootMyb1 scaled STD,0.230,0.250,0.286,0.461,0.572,0.685,0.897
Last time step [s],5500.000,5250.025,5250.025,5500.000,5250.025,5500.000,5250.025


## 10 m/s sawtooth: operating point and phase of the pitch response
The wind oscillates between 9.5 and 10.5 m/s with a 20 s period. Two effects are separated here:

* **Operating point** (mean values): mean pitch, generator torque, mean power and mean TB moment.
* **Phase of the pitch response** (variability): phase-averaged signals over the 20 s wind cycle show whether
  the pitch moves in phase with the wind (as ROSCO does, damping the thrust variation) or out of phase
  (lowering the pitch while the wind rises, which amplifies the thrust and TB oscillation).


In [ ]:
PERIOD = 20.0   # s, period of the sawtooth wind
BIN = 1.0       # s, width of the phase bins

saw10 = SWEEPS["10 m/s sawtooth"]
cycle_runs = {"ROSCO": window(load_fast_file(saw10["rosco"])[0])}
for rate, f in saw10["fixed"].items():
    cycle_runs[f"{rate} °/s"] = window(load_fast_file(f)[0])
cycle_runs["0.004 °/s + offset"] = window(load_fast_file(DIAGNOSTIC)[0])

# Operating point (means) and variability (standard deviations) in physical units
dt = cycle_runs["ROSCO"]["Time"].diff().median()
op = {}
for label, d in cycle_runs.items():
    op[label] = {
        "Pitch mean [deg]": d["BldPitch1"].mean(),
        "Pitch STD [deg]": d["BldPitch1"].std(),
        "Pitch travel [deg/h]": np.abs(np.diff(d["BldPitch1"])).sum() / (len(d) * dt / 3600),
        "GenTq mean [kN·m]": d["GenTq"].mean(),
        "GenTq STD [kN·m]": d["GenTq"].std(),
        "RotSpeed max [rpm]": d["RotSpeed"].max(),
        "Power mean [kW]": d["GenPwr"].mean(),
        "Power max [kW]": d["GenPwr"].max(),
        "TB mean [MN·m]": d["TwrBsMyt"].mean() / 1e3,
        "TB STD [MN·m]": d["TwrBsMyt"].std() / 1e3,
    }
pd.DataFrame(op).round(3)


In [ ]:
def phase_average(d, channels, period=PERIOD, width=BIN):
    """Mean of each channel in bins of `width` seconds within the wind cycle."""
    phase = ((d["Time"] - d["Time"].iloc[0]) % period)
    bins = (phase // width) * width
    return d.groupby(bins)[channels].mean()

CYCLE_CHANNELS = ["Wind1VelX", "BldPitch1", "RotSpeed", "GenPwr", "TwrBsMyt"]
CYCLE_CASES = ["ROSCO", "0.004 °/s", "0.004 °/s + offset", "0.4 °/s"]

cycles = {}
for label in CYCLE_CASES:
    c = phase_average(cycle_runs[label], CYCLE_CHANNELS)
    c["TwrBsMyt"] = c["TwrBsMyt"] / 1e3   # MN·m
    cycles[label] = c
    print(f"== {label}")
    display(c.round(2).T)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
wind = cycles["ROSCO"]["Wind1VelX"]
for label, c in cycles.items():
    axes[0].plot(c.index, c["BldPitch1"], marker="o", ms=3, label=label)
    axes[1].plot(c.index, c["TwrBsMyt"], marker="o", ms=3, label=label)
axes[0].set_ylabel("Pitch angle [deg]")
axes[1].set_ylabel("TB fore-aft moment [MN·m]")
for ax in axes:
    ax.set_xlabel("Time within the 20 s wind cycle [s]")
    ax.grid(alpha=0.3)
    twin = ax.twinx()
    twin.plot(wind.index, wind, color="0.6", ls="--", lw=1)
    twin.set_ylabel("Wind speed [m/s]", color="0.5")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("10 m/s sawtooth: phase-averaged pitch and TB moment (dashed: wind)")
fig.tight_layout()
plt.show()


## 15 m/s sawtooth (full load): adaptive rule

In [4]:
tables["15 m/s sawtooth"]

,ROSCO,0.8/0.3 °/s (adaptive)
GenPwr mean,3369.731,3375.923
GenPwr scaled STD,0.344,0.122
TwrBsMyt mean,29408.855,29548.784
TwrBsMyt scaled STD,0.288,0.249
RootMyb1 mean,3980.567,4000.835
RootMyb1 scaled STD,0.238,0.230
Last time step [s],5500.000,5250.025


## 11 m/s TI = 10 % (design scenario of the adaptive rule)

In [5]:
tables["11 m/s TI=10%"]

,ROSCO,0.2 °/s,0.3 °/s,0.4 °/s,0.6 °/s,0.8 °/s,0.8/0.3 °/s (adaptive)
GenPwr mean,3347.209,3384.168,3374.485,3373.634,3370.617,3369.611,3368.720
GenPwr scaled STD,0.105,0.204,0.115,0.074,0.064,0.069,0.063
TwrBsMyt mean,45099.925,45938.036,45858.975,45857.211,45778.442,45743.865,45764.821
TwrBsMyt scaled STD,0.122,0.199,0.186,0.184,0.205,0.232,0.178
RootMyb1 mean,6384.686,6505.732,6487.777,6486.592,6475.244,6472.677,6472.267
RootMyb1 scaled STD,0.130,0.173,0.165,0.162,0.171,0.184,0.159
Last time step [s],5500.000,5250.025,5250.025,5250.025,5250.025,5250.025,5250.025


## 15 m/s TI = 10 %

In [6]:
tables["15 m/s TI=10%"]

,ROSCO,0.2 °/s,0.4 °/s,0.8 °/s,0.8/0.3 °/s (adaptive)
GenPwr mean,3370.621,3373.669,3369.897,3371.456,3373.565
GenPwr scaled STD,0.137,0.167,0.076,0.053,0.056
TwrBsMyt mean,30183.345,30238.703,30234.369,30266.365,30266.286
TwrBsMyt scaled STD,0.119,0.137,0.143,0.187,0.134
RootMyb1 mean,3888.210,3900.865,3897.007,3902.412,3903.842
RootMyb1 scaled STD,0.148,0.154,0.154,0.164,0.152
Last time step [s],5500.000,5500.000,5500.000,5500.000,5250.025


## Activity of the adaptive rule
Fraction of time at the fast rate (|P − P_rated| > 0.1 MW) and duration of the fast-rate episodes.

In [7]:
rows = []
for scenario, df in adaptive_runs.items():
    d = window(df)
    fast = (np.abs(P_RATED - d["GenPwr"]) > THRESHOLD).to_numpy().astype(int)
    edges = np.diff(np.r_[0, fast, 0])
    durations = (np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)) * d["Time"].diff().median()
    rows.append({"Scenario": scenario,
                 "Time at 0.8 °/s [%]": 100 * fast.mean(),
                 "Fast episodes": len(durations),
                 "Median episode [s]": np.median(durations) if len(durations) else np.nan,
                 "90th pct episode [s]": np.percentile(durations, 90) if len(durations) else np.nan})
pd.DataFrame(rows).set_index("Scenario").round(2)

,Time at 0.8 °/s [%],Fast episodes,Median episode [s],90th pct episode [s]
Scenario,,,,
15 m/s sawtooth,0.07,2,1.80,2.00
11 m/s TI=10%,11.75,206,2.74,5.30
15 m/s TI=10%,7.05,202,1.40,3.69


## Table 5: adaptive rule

In [8]:
rows = {}
for scenario, cfg in SWEEPS.items():
    if not cfg["adaptive"]:
        continue
    t = tables[scenario]
    col = {}
    for ch in CHANNELS:
        col[f"{ch} mean DDQN"] = t.loc[f"{ch} mean", ADAPTIVE]
        col[f"{ch} mean ROSCO"] = t.loc[f"{ch} mean", "ROSCO"]
        col[f"{ch} scaled STD DDQN"] = t.loc[f"{ch} scaled STD", ADAPTIVE]
        col[f"{ch} scaled STD ROSCO"] = t.loc[f"{ch} scaled STD", "ROSCO"]
    rows[scenario] = col
table5 = pd.DataFrame(rows)
table5

,15 m/s sawtooth,11 m/s TI=10%,15 m/s TI=10%
GenPwr mean DDQN,3375.923,3368.720,3373.565
GenPwr mean ROSCO,3369.731,3347.209,3370.621
GenPwr scaled STD DDQN,0.122,0.063,0.056
GenPwr scaled STD ROSCO,0.344,0.105,0.137
TwrBsMyt mean DDQN,29548.784,45764.821,30266.286
TwrBsMyt mean ROSCO,29408.855,45099.925,30183.345
TwrBsMyt scaled STD DDQN,0.249,0.178,0.134
TwrBsMyt scaled STD ROSCO,0.288,0.122,0.119
RootMyb1 mean DDQN,4000.835,6472.267,3903.842
RootMyb1 mean ROSCO,3980.567,6384.686,3888.210


## Data for Fig. 5
* Panel A, 10 m/s sawtooth: fixed rates 0.004, 0.04, 0.4 and 0.6 °/s.
* Panel B, 11 m/s TI = 10 %: fixed rates 0.3–0.8 °/s and the two-level adaptive rule.

Saved to `Figures/fig5_data.csv` and read by `05_paper_figures.ipynb`.

In [9]:
# panel -> scenario, {series: {x: column in tables[scenario]}}
FIG5 = {
    "A": ("10 m/s sawtooth", {
        "fixed": {0.004: "0.004 °/s", 0.04: "0.04 °/s", 0.4: "0.4 °/s", 0.6: "0.6 °/s"},
    }),
    "B": ("11 m/s TI=10%", {
        "fixed": {0.3: "0.3 °/s", 0.4: "0.4 °/s", 0.6: "0.6 °/s", 0.8: "0.8 °/s"},
        "adaptive": {0: ADAPTIVE},
    }),
}

records = []
for panel, (scenario, series) in FIG5.items():
    t = tables[scenario]
    for ch in CHANNELS:
        for kind, columns in {**series, "rosco": {0: "ROSCO"}}.items():
            for x, col in columns.items():
                records.append({"panel": panel, "scenario": scenario, "series": kind, "x": x,
                                "channel": ch, "scaled_std": t.loc[f"{ch} scaled STD", col]})
fig5 = pd.DataFrame(records)
FIG_DIR.mkdir(exist_ok=True)
fig5.to_csv(FIG_DIR / "fig5_data.csv", index=False)
fig5.round(3)

,panel,scenario,series,x,channel,scaled_std
0,A,10 m/s sawtooth,fixed,0.004,GenPwr,0.126
1,A,10 m/s sawtooth,fixed,0.040,GenPwr,0.137
2,A,10 m/s sawtooth,fixed,0.400,GenPwr,0.247
3,A,10 m/s sawtooth,fixed,0.600,GenPwr,0.301
4,A,10 m/s sawtooth,rosco,0.000,GenPwr,0.367
5,A,10 m/s sawtooth,fixed,0.004,TwrBsMyt,0.304
6,A,10 m/s sawtooth,fixed,0.040,TwrBsMyt,0.357
7,A,10 m/s sawtooth,fixed,0.400,TwrBsMyt,0.893
8,A,10 m/s sawtooth,fixed,0.600,TwrBsMyt,1.169
9,A,10 m/s sawtooth,rosco,0.000,TwrBsMyt,0.269
